### Build forward and reverse graph from resources for all resource types

In [1]:
from graph_builder import build_fhir_graphs

# Load FHIR resources and build graphs
forward_graph, reverse_graph, resources_map = build_fhir_graphs()

Processing: Condition
Processing: Encounter
Processing: Encounter
Processing: Location
Processing: Medication
Processing: MedicationAdministration
Processing: MedicationAdministration
Processing: MedicationDispense
Processing: MedicationRequest
Processing: Observation
Processing: Observation
Processing: Observation
Processing: Observation
Processing: Observation
Processing: Observation
Processing: Observation
Processing: Organization
Processing: Patient
Processing: Procedure
Processing: Procedure
Processing: Specimen
Processing: Specimen

Graphs built successfully!
Resource types: ['Condition', 'Encounter', 'Location', 'Medication', 'MedicationAdministration', 'MedicationDispense', 'MedicationRequest', 'Observation', 'Organization', 'Patient', 'Procedure', 'Specimen']
Total resources: 887269


### Create Patient bundles from the graphs.
#### Iterate through the reverse graph, which already have direct references for a patient
#### Extend with the forward references for the indirect refs.

In [2]:
def build_patient_bundle(patient_id, reverse_graph, forward_graph, resources_map):
    bundle_resources = set()
    bundle_resources.add(patient_id)  # Add the patient itself
    
    # Get all direct references to this patient
    direct_refs = reverse_graph.get(patient_id, set())
    bundle_resources.update(direct_refs)
    
    # For each direct reference, get their forward references (indirect refs)
    for ref_id in direct_refs:
        indirect_refs = forward_graph.get(ref_id, set())
        bundle_resources.update(indirect_refs)
    
    return bundle_resources

# Build bundles for all patients
patient_bundles = {}
patient_count = 0

# Find all patient resources
for patient_id, patient_resource in resources_map['Patient'].items():
    patient_count += 1
    bundle = build_patient_bundle(patient_id, reverse_graph, forward_graph, resources_map)
    patient_bundles[patient_id] = {
        'resources': bundle,
        'resource_objects': []
    }
    
    # Get the actual resource objects
    for resource_id in bundle:
        resource_type = resource_id.split('/')[0]
        if resource_type in resources_map and resource_id in resources_map[resource_type]:
            patient_bundles[patient_id]['resource_objects'].append(
                resources_map[resource_type][resource_id]
            )

print(f"Created {len(patient_bundles)} patient bundles")
print(f"Total patients: {patient_count}")

# Show statistics for first patient
first_patient_id = list(patient_bundles.keys())[0]
bundle = patient_bundles[first_patient_id]
print(f"\nFirst patient ({first_patient_id}):")
print(f"\nTotal resources in bundle: {len(bundle['resources'])}")
print(f"\nResource types: {sorted(set(r.split('/')[0] for r in bundle['resources']))}")


Created 100 patient bundles
Total patients: 100

First patient (Patient/0a8eebfd-a352-522e-89f0-1d4a13abdebc):

Total resources in bundle: 1649

Resource types: ['Condition', 'Encounter', 'Location', 'Medication', 'MedicationAdministration', 'MedicationDispense', 'MedicationRequest', 'Observation', 'Organization', 'Patient', 'Procedure', 'Specimen']


#### Persist Patient Bundle FHIR resources

In [3]:
from pathlib import Path
import json

def persist_patient_bundles(patient_bundles, output_dir, resources_map):
    output_dir.mkdir(parents=True, exist_ok=True)
    
    for patient_id, bundle_data in patient_bundles.items():
        patient_uuid = patient_id.split("/")[1]
        resource_objects = bundle_data["resource_objects"]
        
        resource_objects.sort(
            key=lambda r: (
                0 if r["resourceType"] == "Patient" else 1,
                r["resourceType"]
            )
        )
        
        fhir_bundle = {
            "resourceType": "Bundle",
            "type": "collection",
            "entry": [
                {"resource": resource}
                for resource in resource_objects
            ]
        }

        output_file = output_dir / f"{patient_uuid}.json"

        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(
                fhir_bundle,
                f,
                indent=2,
                ensure_ascii=False
            )
    
    return len(patient_bundles)

In [4]:
# Persist all patient bundles
output_dir = Path("input/mimic_fhir_bundles")
persisted = persist_patient_bundles(patient_bundles, output_dir, resources_map)
print(f"Saved {persisted} bundles to {output_dir}")

Saved 100 bundles to input\mimic_fhir_bundles
